In [6]:
import numpy as np
import xarray as xr
import os
import seawater
import matplotlib.pyplot as plt
import cmocean

/var/folders/h3/s4smkhd56ks24x42jbqd3gcc0g4l6k/T/ipykernel_7841/3053879634.py:4: UserWarning: The seawater library is deprecated! Please use gsw instead.
  import seawater


In [7]:
sal_path_vs = '.../sal_vs'
sal_path_vd = '.../sal_vd'
temp_path_vs = '.../temp_vs'
temp_path_vd = '.../temp_vd'

In [3]:
month_outputs = {
    1: range(0, 6),      # January
    2: range(6, 11),     # February
    3: range(11, 18),    # March
    4: range(18, 24),    # April
    5: range(24, 30),    # May
    6: range(30, 36),    # June
    7: range(36, 42),    # July
    8: range(42, 48),    # August
    9: range(48, 54),    # September
    10: range(54, 60),   # October
    11: range(60, 66),   # November
    12: range(66, 73)    # December
}

In [11]:
def calc_convective_resistance(case_conf):

    YS, YE = 2007, 2017
    months = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
    days = [day for month in months for day in month_outputs[month]]
    height = 250.0
    g = 9.81
    rho0 = 1026.0

    mesh_path = '...'
    output_path = '.../Figure2&12_ConvectiverResistance'

    case_dict = {
        "ANHA4-EJM010-S": (sal_path_vs, temp_path_vs, 'vs'),
        "ANHA4-EJM012-S": (sal_path_vd, temp_path_vd, 'vd')
    }

    sal_path, temp_path, simulation = case_dict[case_conf]

    data_mesh = xr.open_dataset(os.path.join(mesh_path, 'ANHA4_mesh_zgr.nc'))

    min_x, min_y = 0, 0
    max_x, max_y = 544, 800

    x_count = max_x - min_x
    y_count = max_y - min_y

    bat = data_mesh['hdept'].values[0]
    bat = bat[min_y:max_y, min_x:max_x]
    e3t = data_mesh['e3t_0'].values[0]
    e3t_ps = data_mesh['e3t_ps'].values[0]
    e3t_ps = e3t_ps[min_y:max_y, min_x:max_x]
    tmask_full = data_mesh['tmask'].values[0]
    tmask_full = tmask_full[:, min_y:max_y, min_x:max_x]
    length = data_mesh['e1t'].values[0, min_y:max_y, min_x:max_x]
    width = data_mesh['e2t'].values[0, min_y:max_y, min_x:max_x]

    cumulative_e3t = np.cumsum(e3t)

    z_count = np.where(cumulative_e3t < height)[0][-1] + 1

    Depth = np.zeros((z_count, y_count, x_count))

    for z in range(z_count - 1):
        wet = tmask_full[z] > 0
        Depth[z, wet] = e3t[z]
        partial = wet & (tmask_full[z + 1] == 0)
        Depth[z, partial] = e3t_ps[partial]

    Tmask = np.where(tmask_full[:z_count] > 0, 1.0, np.nan)
    Hmask = np.zeros_like(Depth)
    cumulative_depth = np.cumsum(Depth, axis=0)
    Hmask[cumulative_depth <= height] = 1

    for year in range(YS, YE + 1):
        conv_r = []
        salinity_data = np.load(f"{sal_path}/{case_conf}_{year}_Salinity.npz")
        temperature_data = np.load(f"{temp_path}/{case_conf}_{year}_Temperature.npz")

        salinity = salinity_data['ConvE']
        temperature = temperature_data['ConvE']

        salinity = salinity[:, :, min_y:max_y, min_x:max_x]
        temperature = temperature[:, :, min_y:max_y, min_x:max_x]

        for day in days:
            pdens = seawater.dens0(salinity[day], temperature[day])
            pdens[pdens < 1000] = np.nan
            pdens = pdens[:z_count]
            MaxDens = pdens * Hmask * Tmask
            MaxDens = np.nanmax(MaxDens, axis=0)
            PotDens = pdens * Hmask * Tmask
            PotDens[PotDens == 0] = np.nan
            
            Inner = np.nansum(PotDens * Depth, axis=0)
            Outer = (np.nansum(Depth, axis=0) * MaxDens)
            Diff = np.abs(Outer - Inner)
            Conv = g * Diff
            Conv[Conv == 0] = np.nan
            Conv[bat < height] = np.nan
            Conv = Conv / rho0
            conv_r.append(Conv)

        if conv_r:

            conv_r_array = np.stack(conv_r)
            save_path = (f"{output_path}/"f"{case_conf}_{year}_CR.npz")
            np.savez(save_path, ConvR=conv_r_array)
            print(f"Saved {year}: "f"{conv_r_array.shape}")

In [12]:
calc_convective_resistance("ANHA4-EJM012-S")

/var/folders/h3/s4smkhd56ks24x42jbqd3gcc0g4l6k/T/ipykernel_7841/2687343754.py:71: RuntimeWarning: All-NaN slice encountered
  MaxDens = np.nanmax(MaxDens, axis=0)


Saved 2007: (73, 800, 544)
Saved 2008: (73, 800, 544)
Saved 2009: (73, 800, 544)
Saved 2010: (73, 800, 544)
Saved 2011: (73, 800, 544)
Saved 2012: (73, 800, 544)
Saved 2013: (73, 800, 544)
Saved 2014: (73, 800, 544)
Saved 2015: (73, 800, 544)
Saved 2016: (73, 800, 544)
Saved 2017: (73, 800, 544)


In [13]:
calc_convective_resistance("ANHA4-EJM010-S")

/var/folders/h3/s4smkhd56ks24x42jbqd3gcc0g4l6k/T/ipykernel_7841/2687343754.py:71: RuntimeWarning: All-NaN slice encountered
  MaxDens = np.nanmax(MaxDens, axis=0)


Saved 2007: (73, 800, 544)
Saved 2008: (73, 800, 544)
Saved 2009: (73, 800, 544)
Saved 2010: (73, 800, 544)
Saved 2011: (73, 800, 544)
Saved 2012: (73, 800, 544)
Saved 2013: (73, 800, 544)
Saved 2014: (73, 800, 544)
Saved 2015: (73, 800, 544)
Saved 2016: (73, 800, 544)
Saved 2017: (73, 800, 544)
